# Thuật toán lặp giá trị
Tác giả: stzhao
github: https://github.com/zhaoshitian

> **Ghi chú của người dịch:** Code trong notebook này đã được chuyển từ `gym` (đã ngừng phát triển) sang **Gymnasium** (`pip install gymnasium`): `env.reset()` trả về `(state, info)`, `env.step()` trả về 5 giá trị `(next_state, reward, terminated, truncated, info)`, và `env.seed()` được thay bằng `env.reset(seed=...)`. Môi trường `envs/simple_grid.py` cũng đã được cập nhật theo giao diện Gymnasium. Output bên dưới vẫn là kết quả chạy bằng `gym` cũ của bản gốc, nên số liệu có thể hơi khác khi chạy lại.


# 1. Định nghĩa môi trường


In [30]:
import numpy as np
import sys,os
curr_path = os.path.abspath('')
parent_path = os.path.dirname(curr_path)
sys.path.append(parent_path)
from envs.simple_grid import DrunkenWalkEnv

In [63]:
def all_seed(env,seed = 1):
    ## hàm này chủ yếu để cố định hạt giống ngẫu nhiên
    import numpy as np
    import random
    import os
    env.reset(seed=seed) # Gymnasium bỏ env.seed(), seed được truyền qua reset()
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed) 


In [32]:
env = DrunkenWalkEnv(map_name="theAlley")
all_seed(env, seed = 1) # đặt hạt giống ngẫu nhiên là 1

# 2. Thuật toán lặp giá trị


In [40]:
def value_iteration(env, theta=0.005, discount_factor=0.9):
    Q = np.zeros((env.nS, env.nA)) # khởi tạo một bảng Q
    count = 0
    while True:
        delta = 0.0
        Q_tmp = np.zeros((env.nS, env.nA))
        for state in range(env.nS):
            for a in range(env.nA):
                accum = 0.0
                reward_total = 0.0
                for prob, next_state, reward, done in env.P[state][a]:
                    accum += prob* np.max(Q[next_state, :])
                    reward_total += prob * reward
                Q_tmp[state, a] = reward_total + discount_factor * accum
                delta = max(delta, abs(Q_tmp[state, a] - Q[state, a]))
        Q = Q_tmp
        
        count += 1
        if delta < theta or count > 100: # ở đây thiết lập: dù thuật toán chưa hội tụ, chạy 100 lần cũng thoát vòng lặp
            break 
    return Q

In [41]:
Q = value_iteration(env)
print(Q)

[[2.25015697e+22 2.53142659e+22 4.50031394e+22 2.53142659e+22]
 [2.81269621e+22 5.41444021e+22 1.01257064e+23 5.41444021e+22]
 [6.32856648e+22 1.21824905e+23 2.27828393e+23 1.21824905e+23]
 [1.42392746e+23 2.74106036e+23 5.12613885e+23 2.74106036e+23]
 [3.20383678e+23 5.76690620e+23 1.15338124e+24 5.76690620e+23]
 [7.20863276e+23 1.38766181e+24 2.59510779e+24 1.38766181e+24]
 [1.62194237e+24 3.12223906e+24 5.83899253e+24 3.12223906e+24]
 [3.64937033e+24 7.02503789e+24 1.31377332e+25 7.02503789e+24]
 [8.21108325e+24 1.47799498e+25 2.95598997e+25 1.47799498e+25]
 [1.84749373e+25 3.55642543e+25 6.65097743e+25 3.55642543e+25]
 [4.15686089e+25 8.00195722e+25 1.49646992e+26 8.00195722e+25]
 [9.35293701e+25 1.80044037e+26 3.36705732e+26 1.80044037e+26]
 [5.89235032e+26 7.36543790e+26 7.57587898e+26 7.36543790e+26]]


In [69]:
policy = np.zeros([env.nS, env.nA]) # khởi tạo một bảng chính sách
for state in range(env.nS):
    best_action = np.argmax(Q[state, :]) #chọn chính sách theo bảng Q có được từ thuật toán lặp giá trị
    policy[state, best_action] = 1

policy = [int(np.argwhere(policy[i]==1)[0][0]) for i in range(env.nS) ] # NumPy 2.x không cho int() trên mảng 2 chiều, nên lấy phần tử [0][0]
print(policy)

[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


# 3. Kiểm thử

In [78]:
num_episode = 1000 # kiểm thử 1000 lần
def test(env,policy):
    
    rewards = []  # ghi lại phần thưởng của mọi tập
    success = []  # ghi lại tập này có đi thành công đến đích hay không
    for i_ep in range(num_episode):
        ep_reward = 0  # ghi lại reward của mỗi episode
        state, info = env.reset()  # đặt lại môi trường, chơi lại một ván (tức bắt đầu một tập mới), ở đây state=0
        while True:
            action = policy[state]  # chọn một hành động theo thuật toán
            next_state, reward, terminated, truncated, _ = env.step(action)  # tương tác một lần với môi trường
            done = terminated or truncated  # Gymnasium tách done thành terminated (kết thúc thật) và truncated (bị cắt do giới hạn số bước)
            state = next_state  # cập nhật trạng thái
            ep_reward += reward
            if done:
                break
        if state==12: # tức đã đến đích
            success.append(1)
        else:
            success.append(0)
        rewards.append(ep_reward)
    acc_suc = np.array(success).sum()/num_episode
    print("Tỉ lệ thành công khi kiểm thử là:", acc_suc)
    

In [79]:
test(env, policy)

Tỉ lệ thành công khi kiểm thử là: 0.64
